# Clase 6 — Entrenar una CNN

Curso **Deep Learning con Python y PyTorch desde cero** · DIH DATAlife (Hub de Innovación Digital Europeo)

**Duración estimada de la clase:** 120 minutos

## Objetivos de aprendizaje

- Adaptar el bucle de entrenamiento a una arquitectura convolucional
- Entrenar una red neuronal convolucional, o CNN (Convolutional Neural Network), sobre un dataset de imágenes en color (CIFAR-10)
- Comparar la precisión de una CNN frente a una red densa equivalente

Este notebook combina teoría (celdas Markdown) y código funcional en PyTorch, listo para ejecutar en Google Colab sin instalar nada adicional (solo necesitas `torch`, `torchvision` y `matplotlib`, que ya vienen preinstalados). Consulta el capítulo 6 del manual del curso para la explicación completa.

## Parte 1 · El ciclo de entrenamiento no cambia

En la clase 4 entrenaste una red densa siguiendo siempre la misma secuencia de cuatro pasos, como quien aprende a lanzar dardos con los ojos vendados:

1. **Forward**: lanzas el dardo, es decir, pasas un lote de datos por el modelo y obtienes una predicción
2. **Loss**: alguien te dice a qué distancia has caído del centro, es decir, calculas la función de pérdida comparando la predicción con la etiqueta real
3. **Backward**: calculas mentalmente hacia qué lado corregir el brazo, es decir, `loss.backward()` calcula el gradiente de la pérdida respecto a cada peso
4. **Step**: ajustas tu postura para el siguiente lanzamiento, es decir, `optimizer.step()` actualiza los pesos del modelo

En la clase 5 construiste una CNN con capas `nn.Conv2d` y de pooling, pero nunca llegaste a entrenarla. Hoy vas a comprobar que el ciclo de cuatro pasos es exactamente el mismo: solo cambia el modelo que le pasas, no la estructura del bucle.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

print('Versión de PyTorch:', torch.__version__)
print('Versión de torchvision:', torchvision.__version__)

## Parte 2 · De blanco y negro a color: CIFAR-10

MNIST son dígitos en escala de grises, centrados y sobre fondo limpio: un tensor de entrada con forma `(lote, 1, 28, 28)`. **CIFAR-10** es un dataset de sesenta mil fotografías en color de diez tipos de objetos reales (avión, automóvil, pájaro, gato, ciervo, perro, rana, caballo, barco y camión), con fondos variados: un tensor de entrada con forma `(lote, 3, 32, 32)`, donde el 3 corresponde a los canales de color rojo, verde y azul, o RGB (Red-Green-Blue).

Cargamos el dataset con `torchvision.datasets.CIFAR10`, que se descarga automáticamente la primera vez que lo usas. Normalizamos los tres canales de color al rango aproximado [-1, 1], igual que normalizaste los píxeles de MNIST en clases anteriores.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class_names = ['avión', 'automóvil', 'pájaro', 'gato', 'ciervo',
               'perro', 'rana', 'caballo', 'barco', 'camión']

print('Imágenes de entrenamiento:', len(train_dataset))
print('Imágenes de prueba:', len(test_dataset))

In [ ]:
def desnormalizar(img_tensor):
    # Revertimos la normalización (0.5, 0.5, 0.5) para poder visualizar la imagen
    img = img_tensor * 0.5 + 0.5
    return img.permute(1, 2, 0).numpy()

imagenes, etiquetas = next(iter(train_loader))

plt.figure(figsize=(9, 3))
for i in range(6):
    plt.subplot(1, 6, i + 1)
    plt.imshow(desnormalizar(imagenes[i]))
    plt.title(class_names[etiquetas[i]])
    plt.axis('off')
plt.show()

## Parte 3 · La misma arquitectura, adaptada a 3 canales

Reutilizamos el patrón de la clase 5 (bloques de convolución + ReLU + pooling, seguidos de capas totalmente conectadas). El único cambio estructural obligatorio: la primera capa `nn.Conv2d` debe aceptar **3 canales de entrada** en vez de 1, porque CIFAR-10 son imágenes en color.

Usamos `padding=1` en cada convolución de 3x3 para que el ancho y el alto de la imagen solo se reduzcan en las capas de pooling. Así el cálculo del tamaño de entrada de la primera capa `nn.Linear` es predecible: partiendo de 32x32, tres bloques de pooling de tamaño 2 la dejan en 4x4, con 128 canales de salida en la última convolución.

In [ ]:
class ConvNetCIFAR(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)    # 3 canales de entrada (RGB)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)  # 10 clases de CIFAR-10

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 32x32 -> 16x16
        x = self.pool(F.relu(self.conv2(x)))   # 16x16 -> 8x8
        x = self.pool(F.relu(self.conv3(x)))   # 8x8 -> 4x4
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


model = ConvNetCIFAR()
print(model)

## Parte 4 · Función de pérdida y optimizador

Igual que en la clase 4, necesitamos una función de pérdida y un optimizador. Como CIFAR-10 es un problema de clasificación con diez clases, usamos `nn.CrossEntropyLoss` (combina internamente softmax con la entropía cruzada) y el optimizador `Adam`, una variante del descenso de gradiente estocástico, o SGD (Stochastic Gradient Descent), que suele converger más rápido en la práctica.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Parte 5 · El bucle de entrenamiento

Este es el mismo bucle de cuatro pasos de la clase 4: `optimizer.zero_grad()` limpia los gradientes del paso anterior, `model(imagenes)` es el forward, `criterion(salidas, etiquetas)` calcula la pérdida, `loss.backward()` calcula los gradientes y `optimizer.step()` actualiza los pesos. Repetimos esta secuencia para cada lote de imágenes, y repetimos todo el recorrido del dataset durante varias épocas.

Entrenar sobre las cincuenta mil imágenes de CIFAR-10 en CPU tarda más que entrenar sobre MNIST: es normal que cada época tarde uno o dos minutos. La clase 7 del curso te enseña a acelerar este mismo bucle usando una GPU (Graphics Processing Unit).

In [ ]:
epocas = 5

for epoca in range(epocas):
    model.train()
    perdida_acumulada = 0.0
    for imagenes, etiquetas in train_loader:
        optimizer.zero_grad()            # limpiamos los gradientes del paso anterior
        salidas = model(imagenes)        # forward
        loss = criterion(salidas, etiquetas)
        loss.backward()                  # backward: calcula los gradientes
        optimizer.step()                 # actualiza los pesos del modelo
        perdida_acumulada += loss.item()

    perdida_media = perdida_acumulada / len(train_loader)
    print(f'Época {epoca + 1}/{epocas} - pérdida media: {perdida_media:.4f}')

## Parte 6 · Evaluación sobre el conjunto de prueba

Con el modelo ya entrenado, lo pasamos a modo evaluación con `model.eval()` y desactivamos el cálculo de gradientes con `torch.no_grad()`, porque durante la evaluación no vamos a actualizar ningún peso. Calculamos la exactitud (accuracy) global comparando la clase predicha (`torch.argmax`) con la etiqueta real, sobre el conjunto de prueba que el modelo nunca vio durante el entrenamiento.

In [ ]:
model.eval()
correctas = 0
total = 0

with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        salidas = model(imagenes)
        predicciones = torch.argmax(salidas, dim=1)
        correctas += (predicciones == etiquetas).sum().item()
        total += etiquetas.size(0)

print(f'Exactitud de la CNN en el conjunto de prueba: {correctas / total:.2%}')

## Parte 7 · Por qué una CNN gana a una red densa equivalente

No vamos a entrenar una red densa completa sobre CIFAR-10 en esta clase (tardaría demasiado y no aportaría nada nuevo), pero sí podemos comparar cuántos parámetros necesitaría cada enfoque solo en su primera capa:

- Una red densa que recibiera las 3.072 entradas de una imagen de CIFAR-10 (32 × 32 × 3) y las conectara a una capa oculta de 256 neuronas necesitaría 3.072 × 256 ≈ 786.000 pesos, uno por cada combinación posible de píxel y neurona. Además, esa red no tiene ninguna noción de que un píxel está "al lado" de otro: si el gato se desplaza unos píxeles en la imagen, para la red densa es un patrón de entrada distinto que tendría que aprender por separado
- La primera capa `nn.Conv2d` de nuestra CNN comparte el mismo filtro en toda la imagen, así que necesita muchos menos parámetros para el mismo trabajo de extracción de rasgos

Puedes comprobarlo tú mismo con el número real de parámetros de la primera capa convolucional:

In [ ]:
parametros_conv1 = sum(p.numel() for p in model.conv1.parameters())
parametros_densa_equivalente = 3072 * 256

print(f'Parámetros de la primera capa Conv2d de la CNN: {parametros_conv1:,}')
print(f'Parámetros de la primera capa de una red densa equivalente: {parametros_densa_equivalente:,}')
print(f'La capa densa necesitaría {parametros_densa_equivalente / parametros_conv1:.0f} veces más parámetros')

## Resumen y siguientes pasos

- El ciclo de entrenamiento (forward, pérdida, backward, `optimizer.step()`) es idéntico para una red densa y para una CNN: solo cambia el modelo, no la estructura del bucle
- CIFAR-10 introduce imágenes en color de forma `(lote, 3, 32, 32)`, frente a los `(lote, 1, 28, 28)` de MNIST: el único cambio obligatorio en la arquitectura es que la primera capa `nn.Conv2d` acepte 3 canales de entrada
- `nn.CrossEntropyLoss` es la función de pérdida habitual para clasificación multiclase, combinada con un optimizador como `Adam`
- Una CNN comparte los mismos filtros en toda la imagen (invarianza a la traslación), así que necesita muchos menos parámetros que una red densa equivalente y suele alcanzar mayor precisión sobre imágenes

Ahora abre el notebook de práctica de esta clase (`clase06_practica_entrenar_una_cnn.ipynb`) para completar tú mismo el bucle de entrenamiento y el cálculo de la exactitud.